# 🎬 LLM Attention Spotlight GIF

This is a simple, fun Jupyter project that creates a GIF explaining an important LLM concept: **attention**.

The GIF shows how each token in a sentence looks back at other tokens with different strengths before the model predicts what comes next.

**No API key needed. No OpenAI setup needed.**

It will save the GIF in your current Jupyter working directory. For you, that should be:

```text
C:\Users\Nimith Narapareddy\OneDrive - CCHOMES\Documents\Python Scripts
```


## 1. Install requirements

Run this once if you do not already have the packages installed.


In [ ]:
!pip install matplotlib numpy pillow


## 2. Create the GIF

Run the cell below. It will generate a file called:

```text
llm_attention_spotlight.gif
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import FancyArrowPatch
from IPython.display import Image, display
from pathlib import Path

# ---------------------------------------------------------
# Fun sentence for the visualization
# ---------------------------------------------------------
tokens = ["The", "robot", "chef", "cooked", "pizza", "because", "it", "was", "hungry"]

# Toy attention weights.
# Rows = current token.
# Columns = tokens it attends to.
# This is hand-crafted for learning, not from a real model.
attention = np.array([
    [1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
    [0.20, 0.80, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
    [0.10, 0.55, 0.35, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
    [0.05, 0.25, 0.30, 0.40, 0.00, 0.00, 0.00, 0.00, 0.00],
    [0.03, 0.08, 0.10, 0.34, 0.45, 0.00, 0.00, 0.00, 0.00],
    [0.02, 0.10, 0.11, 0.12, 0.35, 0.30, 0.00, 0.00, 0.00],
    [0.02, 0.40, 0.32, 0.05, 0.09, 0.07, 0.05, 0.00, 0.00],
    [0.01, 0.20, 0.18, 0.05, 0.06, 0.05, 0.35, 0.10, 0.00],
    [0.01, 0.30, 0.28, 0.04, 0.04, 0.04, 0.20, 0.04, 0.05],
])

# Normalize each row so weights add to 1.
attention = attention / attention.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor("#111827")

x_positions = np.linspace(0.08, 0.92, len(tokens))
y_token = 0.72

def draw_token_box(ax, x, y, text, active=False, strength=0):
    color = "#facc15" if active else plt.cm.viridis(strength)
    text_color = "#111827" if active else "white"
    bbox = dict(
        boxstyle="round,pad=0.45",
        facecolor=color,
        edgecolor="white",
        linewidth=2 if active else 1.1,
        alpha=0.95,
    )
    ax.text(
        x, y, text,
        ha="center", va="center",
        fontsize=13,
        color=text_color,
        fontweight="bold",
        bbox=bbox,
        transform=ax.transAxes,
    )

def animate(frame):
    ax.clear()
    ax.set_facecolor("#111827")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    current = frame % len(tokens)
    weights = attention[current]

    ax.text(
        0.5, 0.95,
        "LLM Attention Spotlight",
        ha="center", va="center",
        fontsize=24,
        color="white",
        fontweight="bold",
        transform=ax.transAxes,
    )

    ax.text(
        0.5, 0.89,
        "Each token looks back at useful context before the model predicts what comes next.",
        ha="center", va="center",
        fontsize=12,
        color="#cbd5e1",
        transform=ax.transAxes,
    )

    # Draw attention arrows first so boxes appear on top.
    for j, w in enumerate(weights):
        if j == current or w < 0.06:
            continue

        start = (x_positions[current], y_token - 0.05)
        end = (x_positions[j], y_token - 0.05)
        curve = 0.25 if j < current else -0.25

        arrow = FancyArrowPatch(
            start, end,
            connectionstyle=f"arc3,rad={curve}",
            arrowstyle="-|>",
            mutation_scale=12 + 18*w,
            linewidth=1 + 8*w,
            color=plt.cm.plasma(w),
            alpha=0.25 + 0.75*w,
            transform=ax.transAxes,
        )
        ax.add_patch(arrow)

    # Draw token boxes.
    for j, token in enumerate(tokens):
        draw_token_box(
            ax,
            x_positions[j],
            y_token,
            token,
            active=(j == current),
            strength=weights[j],
        )

    # Attention bar chart.
    ax.text(
        0.08, 0.47,
        f"Current token: {tokens[current]}",
        ha="left", va="center",
        fontsize=15,
        color="#facc15",
        fontweight="bold",
        transform=ax.transAxes,
    )

    ax.text(
        0.08, 0.42,
        "Attention strength",
        ha="left", va="center",
        fontsize=11,
        color="#cbd5e1",
        transform=ax.transAxes,
    )

    bar_left = 0.08
    bar_top = 0.36
    bar_height = 0.035
    max_width = 0.62

    ranked = sorted(list(enumerate(weights)), key=lambda p: p[1], reverse=True)[:5]
    for row, (j, w) in enumerate(ranked):
        y = bar_top - row * 0.055
        ax.text(
            bar_left,
            y,
            tokens[j],
            ha="left", va="center",
            fontsize=11,
            color="white",
            transform=ax.transAxes,
        )
        ax.add_patch(
            plt.Rectangle(
                (bar_left + 0.12, y - bar_height/2),
                max_width * w,
                bar_height,
                color=plt.cm.viridis(w),
                alpha=0.95,
                transform=ax.transAxes,
            )
        )
        ax.text(
            bar_left + 0.13 + max_width * w,
            y,
            f"{w:.0%}",
            ha="left", va="center",
            fontsize=10,
            color="#cbd5e1",
            transform=ax.transAxes,
        )

    # Fun explanation panel.
    if tokens[current] == "it":
        explanation = '"it" pays strong attention to "robot" and "chef" to resolve what "it" refers to.'
    elif tokens[current] == "pizza":
        explanation = '"pizza" attends to "cooked" because actions and objects are connected.'
    elif tokens[current] == "hungry":
        explanation = '"hungry" looks back at the character tokens, helping the sentence make sense.'
    else:
        explanation = "The highlighted word is gathering context from earlier tokens."

    ax.text(
        0.5, 0.07,
        explanation,
        ha="center", va="center",
        fontsize=13,
        color="#e5e7eb",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#1f2937", edgecolor="#374151"),
        transform=ax.transAxes,
    )

animation = FuncAnimation(fig, animate, frames=len(tokens), interval=1000, repeat=True)

gif_path = Path.cwd() / "llm_attention_spotlight.gif"
animation.save(gif_path, writer=PillowWriter(fps=1))
plt.close(fig)

display(Image(filename=str(gif_path)))
print(f"Saved GIF to: {gif_path}")


## 3. LinkedIn caption

Use this with your GIF:

> I made a tiny Python/Jupyter visualization to explain one of the most important LLM concepts: **attention**.  
>  
> The idea: each word, or token, looks at other tokens with different strengths before the model predicts what comes next.  
>  
> This helped me understand why LLMs are good at using context, resolving references like “it,” and connecting related words in a sentence.  
>  
> Built with Python, NumPy, Matplotlib, and Pillow.  
>  
> #AI #LLM #Python #DataScience #GenerativeAI #MachineLearning
